Generate Collect Earth Online plots for a SIDS country, following the CEO example plot GeoJSON.
Each plot is one cell of the 30m grid for the country's region from `get_gridspec`, so plots are exactly the GeoMAD and
LULC pixels. Plots are stratified by ESA WorldCover loaded onto that grid and mapped to our level 1 classes.

1. Load GADM for the country and buffer 200m (antimeridian fixed).
2. For each grid tile covering the buffered country, allocate N_PLOTS by the tile's share of the country's area
   (EPSG:6933, equal area).
3. Load ESA WorldCover (level 1 classes) onto the tile and choose its plots with `equal_stratified_random`. One tile is loaded at a time to keep memory low. Classes with too few pixels give all they have, so the
   total can be a little under N_PLOTS.
4. Merge the plots from all tiles.
5. Write a zipped shapefile in EPSG:4326 with a square polygon per plot, with PLOTID and ExtraInfo (the country name).
   The class is not written, so it does not bias the interpreters.

In [5]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from antimeridian import fix_polygon
from odc.geo.geom import Geometry

from ldn.grids import get_gridspec
from ldn.random_sampling import random_sampling
from ldn.training_data import LULC_PRODUCTS, get_buffered_country, load_and_prepare, product_nodata_value
from ldn.utils import DEP_COUNTRIES_AND_CODES, TRAINING_DATA_YEAR, WGS84, get_analysis_epsg

In [6]:
# country_of_interest = {"Fiji": "FJI"}
country_of_interest = {"Anguilla": "AIA"}
BUFFER_M = 200  # For buffering country polygons
N_PLOTS = 3_500  # Goal, in practice may be less
EQUAL_AREA_CRS = "EPSG:6933"  # For each tile's share of the country's area
SEED = 0
WORLDCOVER = LULC_PRODUCTS[0]
assert WORLDCOVER["product"] == "esa-worldcover", "Product must be `esa-worldcover`"  # Safeguard

name = next(iter(country_of_interest))
PLOTS_PATH = Path(f"{name}_plots.shp.zip")

In [7]:
region = "pacific" if name in DEP_COUNTRIES_AND_CODES else "non-pacific"  # TODO: Make more robust?
analysis_crs = get_analysis_epsg(region)
grid = get_gridspec(region)
country_wgs84_buffered = get_buffered_country(country_of_interest, analysis_crs, BUFFER_M).dissolve()
geom_country_wgs84_buffered = Geometry(country_wgs84_buffered.to_crs(analysis_crs).geometry.iloc[0], crs=analysis_crs)

country_area = country_wgs84_buffered.to_crs(EQUAL_AREA_CRS).area.sum()

# For each tile of a country, calculate the proportion of the country's area it covers
# and allocate the share of samples accordingly. This prevents memory issues by not loading all tiles at once.
# It also means that tiles without much land get an appropriately smaller number of samples.
# e.g. Fiji has 26 tiles that have very uneven amounts of land.

tile_samples = []
for tile_id, geobox in grid.tiles_from_geopolygon(geom_country_wgs84_buffered):
    # Tile extent in WGS84, split at the antimeridian like the country, so the area in EPSG:6933 is correct
    tile_wgs84 = fix_polygon(geobox.extent.to_crs(WGS84))
    tile_area = country_wgs84_buffered.intersection(tile_wgs84).to_crs(EQUAL_AREA_CRS).area.sum()
    n_tile = round(N_PLOTS * tile_area / country_area)
    if n_tile == 0:
        continue

    try:
        # Load WC for the tile (clipped to country buffered 200m)
        band = load_and_prepare(geobox, country_wgs84_buffered, WORLDCOVER, TRAINING_DATA_YEAR)[
            WORLDCOVER["output_band"]
        ]
        if not (band != product_nodata_value).any():
            print(f"Tile {tile_id}: no valid WorldCover pixels, skipping {n_tile} plots")
            continue

        samples = random_sampling(
            da=band.rename({"y": "latitude", "x": "longitude"}),
            n=n_tile,
            min_sample_n=None,  # Not used by equal stratified random. TODO: make this param optional
            sampling="equal_stratified_random",
            drop_value=product_nodata_value,
            seed=SEED,
        )
        print(
            f"Tile {tile_id}: {tile_area / country_area:.1%} of country area, "
            f"{n_tile} requested, {len(samples)} sampled"
        )
        tile_samples.append(samples)
    except Exception as e:
        print(f"Tile {tile_id}: encountered an error: {e}, skipping {n_tile} plots")

samples = pd.concat(tile_samples, ignore_index=True)
print(samples["lulc"].value_counts().sort_index())

Tile (144, 127): 57.2% of country area, 2001 requested, 2002 sampled
Tile (145, 127): 4.9% of country area, 173 requested, 130 sampled
Tile (144, 128): 17.3% of country area, 605 requested, 529 sampled
Tile (145, 128): 20.6% of country area, 721 requested, 623 sampled
lulc
1    500
2    500
3    315
4    469
5    500
6    500
7    500
Name: count, dtype: int64


In [8]:
# Shuffle so PLOTID order does not follow the class
samples = samples.sample(frac=1, random_state=SEED)
centres = samples.geometry.to_crs(analysis_crs)
plots = gpd.GeoDataFrame(
    {"PLOTID": np.arange(1, len(samples) + 1).astype(str), "ExtraInfo": name},
    geometry=centres.buffer(grid.resolution.x / 2, cap_style="square").to_crs(WGS84).values,
)
plots.to_file(PLOTS_PATH, driver="ESRI Shapefile")
print(f"Wrote {len(plots)} plots to {PLOTS_PATH}")

Wrote 3284 plots to Anguilla_plots.shp.zip
